# Generate a STL from MPRAGE Anatomical Brain data

Calvin Woehrle

July 1, 2026

In [ ]:
from pathlib import Path
import typing
from typing import Any, cast

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
from nibabel.nifti1 import Nifti1Image

from mri_utils import find_mprage
# from nilearn import plotting

NIIX_DIR = Path("niix")

# Find the MPRAGE anatomical scan in the niix directory.
mprage_nii_files = find_mprage(NIIX_DIR)
MPRAGE = mprage_nii_files[0]


# Configure the output directory for STL files.
STL_OUT_DIR = Path("STL")
STL_OUT_DIR.mkdir(exist_ok=True)

img = cast(Nifti1Image, nib.load(MPRAGE))

# data: np.ndarray[typing.Any, np.dtype[np.floating]] = img.get_fdata()  # shape (X, Y, Z) -- a single 3D volume
data = np.asarray(img.dataobj)  # shape (X, Y, Z) -- a single 3D volume

zooms = img.header.get_zooms()

print(f"File:        {MPRAGE.name}")
print(f"Shape:       {img.shape}  (X, Y, Z)")
print(f"Voxel size:  {tuple(round(z, 2) for z in zooms[:3])} mm")
print(f"Orientation: {''.join(nib.aff2axcodes(img.affine))}")
print(f"Intensity:   min={data.min():.0f}  max={data.max():.0f}")

In [ ]:
from scipy import ndimage
from skimage import measure

# Use spacing to consider original voxel size when generating the mesh.
spacing = tuple(float(z) for z in zooms[:3])


def largest_component(mask: np.ndarray) -> np.ndarray:
    """Keep only the largest connected component of a boolean mask."""
    labels, n = ndimage.label(mask)
    if n > 1:
        largest = np.argmax(np.bincount(labels.ravel())[1:]) + 1
        mask = labels == largest
    return mask


def mesh_from_mask(mask: np.ndarray, sigma: float = 1.0, level: float = 0.5):
    """Smooth a binary mask and extract a marching-cubes surface (mm units)."""
    smooth = ndimage.gaussian_filter(mask.astype(float), sigma=sigma)
    verts, faces, _normals, _ = measure.marching_cubes(smooth, level=level, spacing=spacing)
    return verts, faces


# ============ Whole Head Mesh Extraction ============
# Threshold head vs. air, fill interior holes, then keep the largest blob.
head_thr = np.percentile(data[data > 0], 60)
head_mask = ndimage.binary_fill_holes(data > head_thr)
head_mask = largest_component(head_mask)

verts, faces = mesh_from_mask(head_mask, sigma=1.0)
print(f"Head:  {len(verts):,} verts, {len(faces):,} tris  (thr={head_thr:.0f})")


# ============ Brain Mesh Extraction ============
# Rough skull-strip: erode to snap the thin scalp/skull bridge, keep the
# largest interior blob (the brain), then dilate back to the original size.
brain_thr = np.percentile(data[data > 0], 70)
brain_seed = ndimage.binary_erosion(data > brain_thr, iterations=6)
brain_seed = largest_component(brain_seed)
brain_mask = ndimage.binary_dilation(brain_seed, iterations=6)
brain_mask = ndimage.binary_fill_holes(brain_mask)

verts_brain, faces_brain = mesh_from_mask(brain_mask, sigma=1.5)
print(f"Brain: {len(verts_brain):,} verts, {len(faces_brain):,} tris  (thr={brain_thr:.0f})")


# ============ Skull Mesh Extraction ============
# Cortical bone is dark on T1, so segment the skull geometrically: peel the
# outer scalp off the head, then remove the (slightly padded) brain region.
# What remains is the bony shell between scalp and brain.
scalp_peeled = ndimage.binary_erosion(head_mask, iterations=4)   # drop outer scalp
brain_padded = ndimage.binary_dilation(brain_mask, iterations=2)  # small safety margin
skull_mask = scalp_peeled & ~brain_padded
skull_mask = largest_component(skull_mask)

verts_skull, faces_skull = mesh_from_mask(skull_mask, sigma=1.0)
print(f"Skull: {len(verts_skull):,} verts, {len(faces_skull):,} tris")



## Interactive 3D head model

The whole-head surface above is a static render. Here the same marching-cubes
mesh (`verts`, `faces`) is drawn with **Plotly**, so you can **click-drag to
rotate**, scroll to zoom, and right-drag to pan the model in real time.


In [ ]:
import plotly.graph_objects as go

# Reuse the whole-head mesh from the marching-cubes cell above.
x, y, z = verts.T
i, j, k = faces.T

head = go.Mesh3d(
    x=x, y=y, z=z,
    i=i, j=j, k=k,
    color="rgb(217, 191, 179)",
    opacity=1.0,
    lighting=dict(ambient=0.45, diffuse=0.9, specular=0.3, roughness=0.5),
    lightposition=dict(x=200, y=200, z=400),
    flatshading=False,
    name="head",
)

fig = go.Figure(data=[head])
fig.update_layout(
    title="Calvin Woehrle - Interactive 3D head model",
    width=800,
    height=800,
    scene=dict(
        xaxis_title="X (mm)",
        yaxis_title="Y (mm)",
        zaxis_title="Z (mm)",
        aspectmode="data",  # keep true anatomical proportions
    ),
    margin=dict(l=0, r=0, t=40, b=0),
)

# Optionally save a standalone, shareable HTML file:
fig.write_html("head_model.html")

# fig.show()


## Export to STL

The marching-cubes mesh (`verts`, `faces`) is already a watertight triangle
surface, so it can be written directly to an **STL** file for 3D printing or
viewing in any CAD/mesh tool. Coordinates are in millimetres, so the print
comes out life-size.


In [ ]:
def write_binary_stl(path, verts, faces):
    """Write a triangle mesh to a binary STL file (units = mm)."""
    tris = verts[faces]  # (n_faces, 3, 3): each triangle's 3 xyz vertices

    # Per-face normal from the triangle edges.
    normals = np.cross(tris[:, 1] - tris[:, 0], tris[:, 2] - tris[:, 0])
    lengths = np.linalg.norm(normals, axis=1, keepdims=True)
    normals = np.divide(normals, lengths, out=np.zeros_like(normals), where=lengths > 0)

    n = len(faces)
    record = np.zeros(
        n,
        dtype=np.dtype([
            ("normal", "<f4", 3),
            ("v", "<f4", (3, 3)),
            ("attr", "<u2"),
        ]),
    )
    record["normal"] = normals
    record["v"] = tris

    with open(path, "wb") as f:
        f.write(b"MPRAGE head model - Calvin Woehrle".ljust(80, b" "))
        f.write(np.uint32(n).tobytes())
        f.write(record.tobytes())
    return path


stl_path = "head_model.stl"
write_binary_stl(stl_path, verts, faces)
print(f"Wrote {stl_path}: {len(faces):,} triangles, {len(verts):,} vertices")


In [ ]:
# ============== Outputs ================


# ============ Export all three meshes to STL ============
# Reuses write_binary_stl (defined above). Files land in STL_OUT_DIR, mm units.
meshes = {
    "head": (verts, faces),
    "brain": (verts_brain, faces_brain),
    "skull": (verts_skull, faces_skull),
}

for name, (v, f) in meshes.items():
    out_path = STL_OUT_DIR / f"{name}.stl"
    write_binary_stl(out_path, v, f)
    print(f"Wrote {out_path}: {len(f):,} triangles, {len(v):,} vertices")

